[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/stokes_swimmer/stokes_swimmer.ipynb)

# Swimming without inertia

In slow, thick flow, a bacterium in water or a person in honey, nothing coasts: the moment a swimmer stops moving its body, it stops. At every instant the pushes its body gives the fluid balance, with no net force and no net turning. Purcell's scallop theorem follows: a swimmer that moves its body back and forth along one path ends every stroke where it began. To get anywhere it must take its shape around a loop, as Purcell's three-link swimmer does with its two hinges.

The swimmer here is a chain of beads in the plane. Each bead pushes on the fluid, the fluid carries every bead along, and the body moves at exactly the rate that keeps all the pushes in balance.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import concatenate
from examples.animation import save_figure, show_animation
from examples.mechanics.stokes_swimmer import core, render
from examples.mechanics.stokes_swimmer.core import ORIGIN, flow_field, separation, three_link_shape

ga = core.ga
mv = core.mv
Point = ga.gatype.antivector()
Twist = ga.gatype.bivector()
Forque = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
Force = ga.gatype.vector().derive.nondegenerate
Direction = Force.dual().output_gatype

## 1. How the body moves

A push on the fluid at one bead sets all of the fluid moving, less and less with distance: the flow of a force spread over a small blob, a regularized Stokeslet. Over every pair of beads, these flows make one map from the forces the beads push with to the velocities they move at. The body's own motion, a turn about some centre, gives every bead a velocity of its own, and the forces the beads push with add up to a single forque on the body. Both come from one map: the map from the body's rate to the beads' velocities, `rigid`, read backwards across the regressive product as its adjugate, sums the forces into the forque, because a forque paired with a rate is the power the forces deliver to beads moving at that rate. As the swimmer changes its shape, its body must move so that this forque stays zero, and that is one solve for the body's rate of motion.

In the matrix notation of microhydrodynamics, the map from forces to velocities is the beads' Oseen matrix, and the body's rate comes from a three by three resistance matrix.

In [ ]:
HINGE_HALF_SPAN = 0.8
CENTRE_OFFSETS = np.array([-0.5, 0.0, 0.5])
ARM_OFFSETS = np.array([0.35, 0.7, 1.05])
BEAD_RADIUS = 0.14


def stokeslet(separations: Force, radius: float):
    """Fluid velocity (`Direction`) per unit blob force (`Force`) at `separations`."""
    # The blob spreads each force over the bead's size: softening the distance by the radius keeps the
    # flow finite where the force acts.
    r2 = separations.norm_squared() + radius**2                             # [...] Scalar
    # The fluid moves along the force, and further along the separation by as much of the force as
    # points along it; both parts fall off with distance.
    return ((r2 + radius**2) * Force.dual() + separations.dual() * (separations | Force)) / (r2 * r2.square_root())


def mobility(sites: Point, radius: float):
    """The flow each bead's force drives at every other bead: one map from all the beads' forces to all
    their velocities. Its blocks depend on every pair of beads, so they are formed over the beads as a
    batch and assembled into the map between the two fields."""
    beads = sites.batch()                                                   # [..., beads] Point
    pairs = separation(beads[..., :, None], beads[..., None, :])            # [..., beads, beads] Force
    return stokeslet(pairs, radius).field(0, 1)                             # [...] Direction[beads] <- Force[beads]


def swim(sites: Point, shape_velocity: Direction, radius: float):
    """The body-frame rigid rate (`Twist`) and bead forces (`Force[beads]`) of a force- and torque-free swimmer."""
    M = mobility(sites, radius)                                             # [...] Direction[beads] <- Force[beads]
    # With the body's rate open, each bead's commutator with it is that bead's velocity.
    rigid = sites.commutator(Twist)                                         # [...] Direction[beads] <- Twist
    # The forces that carry the beads along with the body at a given rate, summed into the forque on the
    # body: the adjugate carries forces back across the map that carried the rate out to velocities.
    resistance = rigid.adjugate()(M.solve(rigid))                           # [...] Forque <- Twist
    # Changing shape alone would push on the fluid too; the body moves at the rate whose forque cancels it.
    body_rate = -resistance.solve(rigid.adjugate()(M.solve(shape_velocity)))   # [...] Twist
    # The forces the beads push with under both motions together, balanced by construction.
    forces = M.solve(rigid(body_rate) + shape_velocity)                         # [...] Force[beads]
    return body_rate, forces

## 2. Purcell's stroke beside a reciprocal stroke

Two swimmers stroke for three cycles. Purcell's swimmer swings its two hinges a quarter cycle apart, a loop in the space of hinge angles; the other swings both together, back and forth along a line. With nothing coasting, the body's rate is linear in the rate at which the shape changes, so how far a stroke carries the swimmer depends only on the path its shape takes, and going back along a path undoes it. The reciprocal swimmer lunges forward and back and ends every cycle where it started, while Purcell's swimmer steps steadily ahead.

The animation shows Purcell's swimmer with the forces its beads push with and the flow they drive, both swimmers' progress over the cycles, and the two strokes in the space of hinge angles.

In [ ]:
AMPLITUDE = 1.05
CYCLES = 3
STEPS_PER_CYCLE = 60
EVERY = 3
GRID_X = np.linspace(-2.8, 2.8, 25)
GRID_Y = np.linspace(-1.9, 1.9, 17)

steps = CYCLES * STEPS_PER_CYCLE
dt = 2 * np.pi / STEPS_PER_CYCLE
phases = np.arange(steps) * dt                                              # [steps]

angles = np.stack([
    np.stack([AMPLITUDE * np.cos(phases), AMPLITUDE * np.sin(phases)], axis=-1),
    np.stack([AMPLITUDE * np.cos(phases), -0.5 * AMPLITUDE * np.cos(phases)], axis=-1),
], axis=1)                                                                  # [steps, swimmers, 2]
rates = np.stack([
    np.stack([-AMPLITUDE * np.sin(phases), AMPLITUDE * np.cos(phases)], axis=-1),
    np.stack([-AMPLITUDE * np.sin(phases), 0.5 * AMPLITUDE * np.sin(phases)], axis=-1),
], axis=1)                                                                  # [steps, swimmers, 2]

sites, shape_velocity = three_link_shape(HINGE_HALF_SPAN, CENTRE_OFFSETS, ARM_OFFSETS, angles, rates)
body_rate, forces = swim(sites, shape_velocity, BEAD_RADIUS)                # [steps, swimmers] Twist, Force[beads]

# Each step moves the body in its own frame, so it multiplies the pose on the right. A running product
# puts later factors on the left; taking it over the steps' reverses and reversing the result puts them
# on the right.
start_motor = (mv.xy * np.array([0.0, 0.0])).exp()                          # [swimmers] Motor
step_motors = (body_rate * (-dt / 2.0)).exp()                               # [steps, swimmers] Motor
motors = concatenate([start_motor[None], start_motor * step_motors.reverse().cumprod(axis=0).reverse()], axis=0)
positions = motors >> ORIGIN                                                # [steps + 1, swimmers] Point
# The beads and their forces carried from the body's frame into the plane; a force is moved as its
# complement, a direction, and complemented again.
world_sites = motors[:-1] >> sites                                          # [steps, swimmers] Point[beads]
world_forces = (motors[:-1] >> forces.dual()).dual()                        # [steps, swimmers] Force[beads]

# The flow Purcell's swimmer drives, sampled on a grid around it every few steps.
gx, gy = np.meshgrid(GRID_X, GRID_Y, indexing="xy")
grid = mv.point(np.stack([gx, gy], axis=-1))                                # [grid_y, grid_x] Point
frame_slice = slice(0, steps, EVERY)
flow = flow_field(
    grid[None],
    world_sites[frame_slice, 0, None, None],
    world_forces[frame_slice, 0, None, None],
    BEAD_RADIUS,
)                                                                           # [frames, grid_y, grid_x] Direction

save_figure(render.draw_summary(angles, positions, world_sites, world_forces, grid, flow[0]), "stokes_swimmer");

In [ ]:
show_animation(render.animate_swimmers(positions, world_sites, grid, flow, EVERY), "stokes_swimmer", 60)

In [ ]:
# checks
self_mobility = stokeslet(0.0 * mv.x + 0.0 * mv.y, BEAD_RADIUS)
np.testing.assert_allclose((self_mobility - (2.0 / BEAD_RADIUS) * Force.dual()).kernel, 0.0, atol=1e-12)

rigid = sites.commutator(Twist)
np.testing.assert_allclose(rigid.adjugate()(forces).kernel, 0.0, atol=1e-12)

cycle_motors = motors[STEPS_PER_CYCLE::STEPS_PER_CYCLE]                     # [cycles, swimmers] Motor
np.testing.assert_allclose((cycle_motors[:, 1] - start_motor[1]).kernel, 0.0, atol=1e-4)
cycle_centres = positions[STEPS_PER_CYCLE::STEPS_PER_CYCLE]                 # [cycles, swimmers] Point
purcell_x = (separation(cycle_centres[:, 0], ORIGIN) | mv.x).to_array()
step_advances = np.diff(np.concatenate([[0.0], purcell_x]))
assert np.all(np.abs(step_advances) > 0.04)
np.testing.assert_allclose(step_advances, step_advances[0], rtol=1e-2)